# NB5 — Results & Visualization

Loads saved artifacts from NB4 and generates paper-ready figures and tables. No model training — all outputs are derived from pre-computed results.

**Inputs (`data/`):**
- `results/final_results.json` — cascade operating points, OOB scores, Stage 2 standalone metrics
- `results/cascade_results.csv` — full cascade evaluation across FPR budgets
- `results/per_fault_recall.csv` — per-fault-type recall at 1% FPR budget
- `models/stage1_rf.joblib`, `models/stage2_rf.joblib` — trained models
- `features.parquet` + `feature_cols.json` — for feature importance extraction

**Outputs (`data/results/figures/`):**
- `per_fault_recall_bar.png` — per-fault-type recall bar chart
- `feature_importance_s1/s2.png` — top-20 MDI importances for each stage
- `fault_example_*.png` — example time-series per fault type
- LaTeX table snippets printed to stdout

**Prerequisites:** Run NB1 → NB2 → NB3 → NB4 first.

In [36]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)

DATA_DIR    = Path('data')
RESULTS_DIR = DATA_DIR / 'results'
MODELS_DIR  = DATA_DIR / 'models'
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load saved results
results = json.loads((RESULTS_DIR / 'final_results.json').read_text())
casc_df = pd.read_csv(RESULTS_DIR / 'cascade_results.csv')
pf_df   = pd.read_csv(RESULTS_DIR / 'per_fault_recall.csv')

# Load models
rf_s1 = joblib.load(MODELS_DIR / 'stage1_rf.joblib')
rf_s2 = joblib.load(MODELS_DIR / 'stage2_rf.joblib')

# Load feature metadata
feat_ref     = json.loads((DATA_DIR / 'feature_cols.json').read_text())
FEATURE_COLS = feat_ref['feature_cols']

print('Loaded results and models.')
print(f'Cascade operating points:')
print(casc_df.to_string(index=False))
print(f'\nPer-fault-type recall (1% FPR):')
print(pf_df.to_string(index=False))

Loaded results and models.
Cascade operating points:
fpr_budget  threshold  attack_recall  fault_recall  normal_fpr
      0.5%      0.631         0.7503        0.4764      0.0048
      1.0%      0.530         0.8845        0.6933      0.0099
      2.0%      0.418         0.9540        0.8087      0.0196

Per-fault-type recall (1% FPR):
           fault_type  n_test_rows  recall
                 bias         5402  0.7873
                drift         9137  0.8130
 intermittent_dropout        13354  0.7301
precision_degradation         3895  0.9897
             stuck_at         3765  0.9201


## 1. Per-Fault-Type Recall Bar Chart

Bar chart of cascade fault recall broken down by fault type at the **2% FPR** operating point.
Recomputes predictions from saved models at the 2% threshold (t=0.418) rather than reading the
stale CSV, so this cell is self-contained. Bars colored **steelblue ≥ 50%** / **red < 50%**.

In [37]:
# Load test features and recompute per-fault recall at 2% FPR from saved models
df_feat   = pd.read_parquet(DATA_DIR / 'features.parquet')
test_df   = df_feat[df_feat['split'] == 'test'].copy()
X_test_pf = test_df[FEATURE_COLS].values
y_test_pf = test_df['label'].values
ft_test_pf = test_df['fault_type'].values if 'fault_type' in test_df.columns else None

t_2pct  = float(casc_df.loc[casc_df['fpr_budget'] == '2.0%', 'threshold'].iloc[0])
p_s1_pf = rf_s1.predict_proba(X_test_pf)[:, 1]
p_s2_pf = rf_s2.predict_proba(X_test_pf)[:, 1]

pred_pf = np.zeros(len(y_test_pf), dtype=np.int8)
anomaly_flag = p_s1_pf >= t_2pct
pred_pf[anomaly_flag] = np.where(p_s2_pf[anomaly_flag] >= 0.5, 1, 2)

pf_rows = []
fault_mask = y_test_pf == 2
for ft in sorted(set(ft_test_pf[fault_mask])):
    if not ft or (isinstance(ft, float) and np.isnan(ft)):
        continue
    mask = fault_mask & (ft_test_pf == ft)
    n    = int(mask.sum())
    pf_rows.append({'fault_type': ft, 'n_test_rows': n,
                    'recall': round(int((pred_pf[mask] == 2).sum()) / n, 4) if n else 0})

pf_df_2pct = pd.DataFrame(pf_rows)
print(f'Per-Fault-Type Recall at 2% FPR (t={t_2pct:.3f}):')
print(pf_df_2pct.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
labels  = pf_df_2pct['fault_type'].tolist()
recalls = pf_df_2pct['recall'].tolist()
colors  = ['steelblue' if r >= 0.5 else 'firebrick' for r in recalls]

bars = ax.bar(labels, recalls, color=colors, edgecolor='white', linewidth=0.5)
for bar, r in zip(bars, recalls):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{r:.1%}', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, 1.1)
ax.set_ylabel('Recall', fontsize=12)
ax.set_xlabel('Fault Type', fontsize=12)
ax.set_title('Per-Fault-Type Recall at 2% FPR Budget', fontsize=13)
ax.tick_params(axis='x', rotation=15)
ax.axhline(y=0.5, color='gray', linestyle='--', lw=0.8, alpha=0.6)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

out = FIGURES_DIR / 'per_fault_recall_bar.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out}')

Per-Fault-Type Recall at 2% FPR (t=0.418):
           fault_type  n_test_rows  recall
                 bias         5402  0.7873
                drift         9137  0.8130
 intermittent_dropout        13354  0.7301
precision_degradation         3895  0.9897
             stuck_at         3765  0.9201
Saved: data/results/figures/per_fault_recall_bar.png


## 1b. Confusion Matrix

Three-class confusion matrix at the 2% FPR operating point, recomputed from saved models.

In [38]:
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

df_feat   = pd.read_parquet(DATA_DIR / 'features.parquet')
test_df   = df_feat[df_feat['split'] == 'test'].copy()
X_test    = test_df[FEATURE_COLS].values
y_test    = test_df['label'].values

t_2pct       = float(casc_df.loc[casc_df['fpr_budget'] == '2.0%', 'threshold'].iloc[0])
p_s1         = rf_s1.predict_proba(X_test)[:, 1]
p_s2         = rf_s2.predict_proba(X_test)[:, 1]
pred         = np.zeros(len(y_test), dtype=np.int8)
anomaly_flag = p_s1 >= t_2pct
pred[anomaly_flag] = np.where(p_s2[anomaly_flag] >= 0.5, 1, 2)

cm      = sk_confusion_matrix(y_test, pred, labels=[0, 1, 2])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Row-normalized fraction')

class_labels = ['Normal', 'Attack', 'Fault']
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(class_labels)
ax.set_yticklabels(class_labels)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Cascade Confusion Matrix', fontsize=12)

for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm_norm[i,j]:.1%}\n({cm[i,j]:,})',
                ha='center', va='center', fontsize=10,
                color='white' if cm_norm[i,j] > 0.5 else 'black')

fig.tight_layout()
out = FIGURES_DIR / 'confusion_matrix.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {out}')
print(f'Threshold: {t_2pct}')

Saved: data/results/figures/confusion_matrix.png
Threshold: 0.418


## 2. Feature Importance — Stage 1 & Stage 2

Top-20 MDI (mean decrease in impurity) feature importances for each stage of the cascade:

- **Stage 1** — reveals which sensor statistics best separate normal from anomalous behavior
- **Stage 2** — reveals which features distinguish cyber attacks from sensor faults

Feature names are formatted as `sensor | statistic_window` (e.g., `2_LT_001_PV | slope_120s`).

In [39]:
TOP_N = 20

for stage, rf, name in [(1, rf_s1, 'stage1'), (2, rf_s2, 'stage2')]:
    importances = rf.feature_importances_
    idx = np.argsort(importances)[::-1][:TOP_N]
    top_feats  = [FEATURE_COLS[i] for i in idx]
    top_imps   = importances[idx]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(TOP_N), top_imps[::-1], color='steelblue', edgecolor='white')
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels([f.replace('__', ' | ') for f in top_feats[::-1]], fontsize=9)
    ax.set_xlabel('Feature Importance (MDI)', fontsize=11)
    ax.set_title(f'Stage {stage} Top-{TOP_N} Feature Importances', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()

    out = FIGURES_DIR / f'feature_importance_{name}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out}')

Saved: data/results/figures/feature_importance_stage1.png
Saved: data/results/figures/feature_importance_stage2.png


## 3. Fault Example Time-Series

One example per fault type from the test split, showing the primary injection sensor. Each plot includes a pre-fault baseline window (gray), the fault window (colored), and a post-fault return to normal — illustrating the visual signature that rolling features are designed to capture.

In [40]:
df_faulted = pd.read_parquet(DATA_DIR / 'wadi_faulted.parquet')

fault_types = ['bias', 'drift', 'intermittent_dropout', 'precision_degradation', 'stuck_at']
fault_colors = {
    'bias': 'steelblue', 'drift': 'darkorange', 'intermittent_dropout': 'firebrick',
    'precision_degradation': 'purple', 'stuck_at': 'green',
}

used_sensors = set()  # each sensor appears in at most one panel

for ft in fault_types:
    ft_rows = df_faulted[df_faulted['fault_type'] == ft]

    # For dropout: prefer burst_dropout events (one large contiguous NaN block).
    if ft == 'intermittent_dropout' and 'fault_mode' in ft_rows.columns:
        burst_rows = ft_rows[ft_rows['fault_mode'] == 'burst_dropout']
        if not burst_rows.empty:
            ft_rows = burst_rows

    if ft_rows.empty:
        print(f'{ft}: no rows')
        continue

    # Score events by median / std of pre-fault baseline. Skip already-used sensors.
    events = ft_rows[['fault_sensor', 'fault_start', 'fault_end']].drop_duplicates()
    best_sensor, best_start, best_end, best_score = None, None, None, -np.inf
    sensor_data_cache = {}

    for _, ev in events.iterrows():
        ev_sensor = ev['fault_sensor']
        if ev_sensor in used_sensors:
            continue
        if ev_sensor not in sensor_data_cache:
            sensor_data_cache[ev_sensor] = df_faulted.sort_values('timestamp')
        sd = sensor_data_cache[ev_sensor]
        pre_vals = sd.loc[sd['timestamp'] < ev['fault_start'], ev_sensor].iloc[-200:]
        if len(pre_vals) == 0:
            continue
        med = float(pre_vals.median())
        std = float(pre_vals.std())
        score = med / std if std > 1e-6 else med
        if score > best_score:
            best_score, best_sensor, best_start, best_end = score, ev_sensor, ev['fault_start'], ev['fault_end']

    if best_sensor is None:
        print(f'{ft}: no unused sensor available')
        continue

    used_sensors.add(best_sensor)
    sensor, start, end = best_sensor, best_start, best_end
    print(f'{ft}: selected sensor={sensor}  score(median/std)={best_score:.2f}')

    # Pull a window: 200 rows before + fault window + 200 rows after
    all_data   = df_faulted.sort_values('timestamp').reset_index(drop=True)
    event_mask = (all_data['timestamp'] >= start) & (all_data['timestamp'] <= end)
    event_idx  = all_data.index[event_mask]
    if len(event_idx) == 0:
        continue
    pos_start = int(event_idx[0])
    pos_end   = int(event_idx[-1])
    window    = all_data.iloc[max(0, pos_start - 200): min(len(all_data), pos_end + 200)]

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(range(len(window)), window[sensor].values, color='gray', lw=0.8, label='normal')

    fault_pos  = [i for i, idx in enumerate(window.index) if idx in set(event_idx)]
    fault_vals = window.iloc[fault_pos][sensor].values if fault_pos else np.array([])

    if fault_pos:
        ax.plot(fault_pos, fault_vals, color=fault_colors.get(ft, 'red'), lw=1.2, label=ft)

        if ft == 'intermittent_dropout':
            nan_pos  = [fault_pos[i] for i, v in enumerate(fault_vals) if np.isnan(v)]
            y_marker = float(np.nanmedian(window[sensor].values))
            if nan_pos:
                ax.scatter(nan_pos, [y_marker] * len(nan_pos),
                           marker='x', color=fault_colors['intermittent_dropout'],
                           s=25, linewidths=1.0, label='dropout (NaN)', zorder=3)

    ax.set_xlabel('Row index', fontsize=10)
    ax.set_ylabel(sensor, fontsize=10)
    ax.set_title(f'Fault example: {ft} on {sensor}', fontsize=11)
    ax.legend(fontsize=9)
    fig.tight_layout()

    out = FIGURES_DIR / f'fault_example_{ft}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out}')


bias: selected sensor=2_LT_001_PV  score(median/std)=342.99
Saved: data/results/figures/fault_example_bias.png
drift: selected sensor=3_LT_001_PV  score(median/std)=48.51
Saved: data/results/figures/fault_example_drift.png
intermittent_dropout: selected sensor=2_PIT_002_PV  score(median/std)=34.22
Saved: data/results/figures/fault_example_intermittent_dropout.png
precision_degradation: selected sensor=2_LT_002_PV  score(median/std)=28.44
Saved: data/results/figures/fault_example_precision_degradation.png
stuck_at: selected sensor=1_LT_001_PV  score(median/std)=5.56
Saved: data/results/figures/fault_example_stuck_at.png


## 4. Paper Tables (LaTeX)

Renders the cascade results and per-fault recall tables as LaTeX snippets for direct inclusion in the paper. Also prints key scalar metrics: Stage 1/2 OOB scores and Stage 2 standalone upper-bound performance.

In [41]:
print('=== Cascade Results Table (LaTeX) ===')
print(casc_df.to_latex(index=False, float_format='%.4f'))

print('\n=== Per-Fault Recall Table (LaTeX) ===')
print(pf_df.to_latex(index=False, float_format='%.4f'))

print('\n=== Key Numbers ===')
print(f"Stage 1 OOB: {results['stage1_oob_score']:.4f}")
print(f"Stage 2 OOB: {results['stage2_oob_score']:.4f}")
s2 = results['stage2_standalone']
print(f"Stage 2 standalone — attack recall: {s2['attack_recall']:.4f}  fault recall: {s2['fault_recall']:.4f}")
for pt in results['cascade_operating_points']:
    print(f"  {pt['fpr_budget']:>5}  fault={pt['fault_recall']:.4f}  attack={pt['attack_recall']:.4f}")

=== Cascade Results Table (LaTeX) ===
\begin{tabular}{lrrrr}
\toprule
fpr_budget & threshold & attack_recall & fault_recall & normal_fpr \\
\midrule
0.5% & 0.6310 & 0.7503 & 0.4764 & 0.0048 \\
1.0% & 0.5300 & 0.8845 & 0.6933 & 0.0099 \\
2.0% & 0.4180 & 0.9540 & 0.8087 & 0.0196 \\
\bottomrule
\end{tabular}


=== Per-Fault Recall Table (LaTeX) ===
\begin{tabular}{lrr}
\toprule
fault_type & n_test_rows & recall \\
\midrule
bias & 5402 & 0.7873 \\
drift & 9137 & 0.8130 \\
intermittent_dropout & 13354 & 0.7301 \\
precision_degradation & 3895 & 0.9897 \\
stuck_at & 3765 & 0.9201 \\
\bottomrule
\end{tabular}


=== Key Numbers ===
Stage 1 OOB: 0.9995
Stage 2 OOB: 1.0000
Stage 2 standalone — attack recall: 1.0000  fault recall: 0.9875
   0.5%  fault=0.4764  attack=0.7503
   1.0%  fault=0.6933  attack=0.8845
   2.0%  fault=0.8087  attack=0.9540
